# Hedonic Pricing

We often try to predict the price of an asset from its observable characteristics. This is generally called **hedonic pricing**: How do the unit's characteristics determine its market price?

1. Clean the data `pierce_county_house_sales.csv` and perform some EDA and visualization to get to know the data set.
2. Transform your variables - particularly categorical ones - for use in your regression analysis.
3. Implement an 80/20 train-test split. Put the test data aside.
4. Build some simple linear models that include no transformations or interactions. Fit them, and determine their RMSE and $R^2$ on the both the training and test sets. Which of your models does the best?
5. Make partial correlation plots for each of the numeric variables in your model. Do you notice any significant non-linearities? 
6. Include transformations and interactions of your variables, and build a more complex model that reflects your ideas about how the features of the asset determine its value. Determine its RMSE and $R^2$ on the training and test sets. How does the more complex model your build compare to the simpler ones?
7. Summarize your results from 1 to 6. Have you learned anything about overfitting and underfitting, or model selection?
8. Use the `sklearn.linear_model.Lasso` to regularize your model and select the most predictive features. Which does it select? What are the RMSE and $R^2$? We'll cover the Lasso later in detail in class.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Amran Belete, Nate Park, Amelia Chen, Lawson Pham

**Date:** 11/9/2026

In [1]:
# Your Phase 1 solution to the problem goes here.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df = pd.read_csv('data/pierce_county_house_sales.csv', low_memory=False)
print("df.shape:")
print(df.shape, '\n')

print("df.dtypes:")
print(df.dtypes, '\n')

df.head()

df.shape:
(16814, 19) 

df.dtypes:
sale_date                        str
sale_price                     int64
house_square_feet              int64
attic_finished_square_feet     int64
basement_square_feet           int64
attached_garage_square_feet    int64
detached_garage_square_feet    int64
fireplaces                     int64
hvac_description                 str
exterior                         str
interior                         str
stories                        int64
roof_cover                       str
year_built                     int64
bedrooms                       int64
bathrooms                      int64
waterfront_type                  str
view_quality                     str
utility_sewer                    str
dtype: object 



,sale_date,sale_price,house_square_feet,attic_finished_square_feet,basement_square_feet,attached_garage_square_feet,detached_garage_square_feet,fireplaces,hvac_description,exterior,interior,stories,roof_cover,year_built,bedrooms,bathrooms,waterfront_type,view_quality,utility_sewer
0,2020-01-02,380000,2157,0,0,528,0,0,Warm and Cool Air Zone,Frame Siding,Drywall,2,Composition Shingle,1993,3,3,NaN,NaN,SEWER/SEPTIC INSTALLED
1,2020-01-02,383000,2238,0,0,396,0,0,Forced Air,Frame Siding,Drywall,2,Composition Shingle,2017,4,3,NaN,NaN,SEWER/SEPTIC INSTALLED
2,2020-01-01,419900,1452,0,0,0,0,1,Forced Air,Frame Siding,Drywall,1,Built Up Rock,1965,3,2,NaN,NaN,SEWER/SEPTIC INSTALLED
3,2020-01-02,251660,832,0,0,0,0,0,Forced Air,Frame Vinyl,Drywall,1,Composition Shingle,1919,2,1,NaN,NaN,SEWER/SEPTIC INSTALLED
4,2020-01-02,500000,1780,0,384,484,0,1,Forced Air,Frame Siding,Drywall,1,Composition Shingle,1955,3,2,NaN,NaN,SEWER/SEPTIC INSTALLED


1. Clean the data `pierce_county_house_sales.csv` and perform some EDA and visualization to get to know the data set.


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

target = "sale_price"


numeric_features = [
    "house_square_feet", "attic_finished_square_feet", "basement_square_feet",
    "attached_garage_square_feet", "detached_garage_square_feet", "fireplaces",
    "stories", "year_built", "bedrooms", "bathrooms",
]

categorical_features = [
    "hvac_description", "exterior", "interior", "roof_cover",
    "waterfront_type", "view_quality", "utility_sewer",
]


# Parse dates and numeric columns; treat blank category strings as missing.
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")

for column in numeric_features + [target]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in categorical_features:
    df[column] = df[column].astype("object").map(
        lambda value: value.strip()
        if isinstance(value, str) and value.strip()
        else np.nan
    )

rows_before = len(df)
df = df.drop_duplicates()
df = df.loc[df[target].gt(0) & df["house_square_feet"].gt(0)].copy()


print(f"Rows retained: {len(df):,} of {rows_before:,}")
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nSale price summary:")
print(df[target].describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(0))
print("\nNumber of category levels:")
print(df[categorical_features].nunique().sort_values(ascending=False))


Rows retained: 16,768 of 16,814
Dataset shape: (16768, 19)

Missing values:
waterfront_type                16216
view_quality                   15852
roof_cover                         8
hvac_description                   7
interior                           7
exterior                           1
sale_date                          0
house_square_feet                  0
sale_price                         0
detached_garage_square_feet        0
dtype: int64

Sale price summary:
count      16768.0
mean      461205.0
std       236116.0
min         2000.0
1%        113224.0
25%       348000.0
50%       416725.0
75%       523882.0
95%       804701.0
99%      1265000.0
max      6130000.0
Name: sale_price, dtype: float64

Number of category levels:
exterior            16
hvac_description     9
view_quality         8
roof_cover           6
utility_sewer        5
waterfront_type      4
interior             2
dtype: int64


2. Transform your variables - particularly categorical ones - for use in your regression analysis.


In [3]:
X = df[numeric_features + categorical_features]
y = df[target]

3. Implement an 80/20 train-test split. Put the test data aside.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")


Train rows: 13,414; test rows: 3,354


4. Build some simple linear models that include no transformations or interactions. Fit them, and determine their RMSE and $R^2$ on the both the training and test sets. Which of your models does the best?


In [6]:
def make_preprocessor(include_numeric=True, include_categories=True):
    transformers = []

    if include_numeric:
        transformers.append(
            ("numeric", SimpleImputer(strategy="median"), numeric_features)
        )

    if include_categories:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(
            ("categorical", categorical_pipeline, categorical_features)
        )

    return ColumnTransformer(transformers, remainder="drop")

def score_model(model, X_train, y_train, X_test, y_test):
    results = []
    for split, features, actual in [
        ("train", X_train, y_train),
        ("test", X_test, y_test),
    ]:
        predicted = model.predict(features)
        results.append({
            "split": split,
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "R2": r2_score(actual, predicted),
        })
    return pd.DataFrame(results).set_index("split")

simple_models = {
    "Numeric only": Pipeline([
        ("preprocessor", make_preprocessor( include_numeric=True, include_categories=False)),
        ("model", LinearRegression()),
    ]),
    "Categorical only": Pipeline([
        ("preprocessor", make_preprocessor(include_numeric=False, include_categories=True)),
        ("model", LinearRegression()),
    ]),
    "Numeric + categorical": Pipeline([
        ("preprocessor", make_preprocessor( include_numeric=True, include_categories=True)),
        ("model", LinearRegression()),
    ]),
}


simple_scores = {}
for name, model in simple_models.items():
    model.fit(X_train, y_train)
    simple_scores[name] = score_model(model, X_train, y_train, X_test, y_test)
    print(f"\n{name}")
    display(simple_scores[name].round(3))

best_simple = min(simple_scores, key=lambda name: simple_scores[name].loc["test", "RMSE"])
print("Best simple model by test RMSE:", best_simple)




Numeric only


,RMSE,R2
split,,
train,185345.196,0.401
test,170421.990,0.411



Categorical only


,RMSE,R2
split,,
train,192675.276,0.353
test,185518.190,0.302



Numeric + categorical


,RMSE,R2
split,,
train,161581.134,0.545
test,155462.840,0.510


Best simple model by test RMSE: Numeric + categorical


5. Make partial correlation plots for each of the numeric variables in your model. Do you notice any significant non-linearities? 


6. Include transformations and interactions of your variables, and build a more complex model that reflects your ideas about how the features of the asset determine its value. Determine its RMSE and $R^2$ on the training and test sets. How does the more complex model your build compare to the simpler ones?


7. Summarize your results from 1 to 6. Have you learned anything about overfitting and underfitting, or model selection?


8. Use the `sklearn.linear_model.Lasso` to regularize your model and select the most predictive features. Which does it select? What are the RMSE and $R^2$? We'll cover the Lasso later in detail in class.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]